[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kewel-corp/book-vol1/blob/main/notebooks/ch24_code.ipynb)

# 第24章　診断・分類とEfficientNet・Transformer：画像にラベルを与える

**『医療診断支援AI開発　第1巻　入門編 ― 動かして、自分で作る（第1巻　入門編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-vol1

## 24.7　コードで動かす：事前学習済みEfficientNetを微調整する

In [ ]:
import timm, torch, torch.nn as nn

# 事前学習済み EfficientNet-B0（まずB0で評価し、必要に応じて B1〜B3 と比較）。出力層を自分のクラス数(例:2=正常/骨折)に合わせて置き換える
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=2)

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 5.0]))  # 少数派(骨折)の誤りに大きな重みを付ける
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for images, labels in train_loader:          # DataLoaderから取り出す(第16章)
    optimizer.zero_grad()
    logits = model(images)                    # 順伝播 (B, 2)
    loss   = criterion(logits, labels)        # どれだけ間違えたか
    loss.backward()                           # 勾配を計算(自動微分)
    optimizer.step()                          # パラメータを更新